# Atraktor jako "punkt idealny" w 2 wymiarach
Mamay 2 cechy: novelty - innowacyjność, feasibility - wykonalność
Atraktor A = (N,F)
Kot jest najbliżej idei którą uważamy za pożądaną.
Atrkator nie przewiduje -> przyciąga przez podobieństwo

In [1]:
import pandas as pd
import numpy as np

In [2]:
projects = pd.DataFrame({
    "name":["P1","P2","P3","P4","P5"],
    "novelty":[0.95,0.70,0.40,0.88,0.60],
    "feasibility":[0.78,0.92,0.45,0.60,0.75]
})

In [5]:
attractor = np.array([0.90,0.80]) #ideał -> (N,F)

In [9]:
def distance_to_attractor(row):
  point = np.array([row["novelty"],row["feasibility"]])
  return np.linalg.norm(point - attractor)

In [10]:
projects["distance"] = projects.apply(distance_to_attractor,axis=1)
projects["resonance"] = 1/(1+projects["distance"])


In [11]:
print(projects.sort_values("resonance",ascending=False))

  name  novelty  feasibility  distance  resonance
0   P1     0.95         0.78  0.053852   0.948900
3   P4     0.88         0.60  0.200998   0.832641
1   P2     0.70         0.92  0.233238   0.810873
4   P5     0.60         0.75  0.304138   0.766790
2   P3     0.40         0.45  0.610328   0.620992


# Atraktor ważony - nie każda cecha równoważna
atraktor zaczyna mieć  strukturę priorytetów

In [13]:
#definicje wag
weights = np.array([1.0,2.0])

In [14]:
def weighted_distance(row):
  point = np.array([row["novelty"],row["feasibility"]])
  diff = point - attractor
  return np.sqrt(np.sum(weights*diff**2))

In [15]:
projects["weighted_distance"] = projects.apply(weighted_distance,axis=1)
projects["weighted_resonance"] = 1/(1+projects["weighted_distance"])

In [16]:
print(projects.sort_values("weighted_resonance",ascending=False))

  name  novelty  feasibility  distance  resonance  weighted_distance  \
0   P1     0.95         0.78  0.053852   0.948900           0.057446   
1   P2     0.70         0.92  0.233238   0.810873           0.262298   
3   P4     0.88         0.60  0.200998   0.832641           0.283549   
4   P5     0.60         0.75  0.304138   0.766790           0.308221   
2   P3     0.40         0.45  0.610328   0.620992           0.703562   

   weighted_resonance  
0            0.945675  
1            0.792206  
3            0.779090  
4            0.764397  
2            0.587005  


# Atraktor dynamiczny - punkt docelowy może się przesuwać.
aktualizacja atraktora na podstawie najlepszych przykładów.
strojenie systmu  - analiza top 3 przypadków i przesunięcie atraktora w ich kierunku.

In [17]:
feature_names = ["novelty","feasibility"]

In [19]:
def weighted_distance_att(row,target):
  point = row[feature_names].to_numpy(dtype=float)
  diff = point - target
  return np.sqrt(np.sum(weights*diff**2))

In [20]:
#ranking statyczny
projects["distance_v1"] = projects.apply(lambda row: weighted_distance_att(row,attractor),axis=1)
projects["resonance_v1"] = 1/(1+projects["distance_v1"])
ranked = projects.sort_values("resonance_v1",ascending=False)

In [21]:
#bierzemy top3 przypadków i przesuwamy atraktor w ich kierunku
top3_mean = ranked.head(3)[feature_names].mean().to_numpy()
alpha = 0.7
new_attractor = alpha*attractor+(1-alpha)*top3_mean

In [22]:
#ranking dynamiczny
projects["distance_v2"] = projects.apply(lambda row: weighted_distance_att(row,new_attractor),axis=1)
projects["resonance_v2"] = 1/(1+projects["distance_v2"])

In [23]:
print(f"stary attractor: {attractor}")
print(f"nowy attractor: {new_attractor}")
print(projects.sort_values("resonance_v2",ascending=False))

stary attractor: [0.9 0.8]
nowy attractor: [0.883 0.79 ]
  name  novelty  feasibility  distance  resonance  weighted_distance  \
0   P1     0.95         0.78  0.053852   0.948900           0.057446   
1   P2     0.70         0.92  0.233238   0.810873           0.262298   
3   P4     0.88         0.60  0.200998   0.832641           0.283549   
4   P5     0.60         0.75  0.304138   0.766790           0.308221   
2   P3     0.40         0.45  0.610328   0.620992           0.703562   

   weighted_resonance  distance_v1  resonance_v1  distance_v2  resonance_v2  
0            0.945675     0.057446      0.945675     0.068476      0.935912  
1            0.792206     0.262298      0.792206     0.259401      0.794028  
3            0.779090     0.283549      0.779090     0.268717      0.788198  
4            0.764397     0.308221      0.764397     0.288598      0.776037  
2            0.587005     0.703562      0.587005     0.681534      0.594695  
